In [1]:
!pip install guardrails-ai presidio-analyzer presidio-anonymizer https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.1 MB/s  0:00:00eta 0:00:01


In [2]:
!pip install spacy spacy-transformers

In [3]:
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 78.0 MB/s  0:00:04:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
!pip install -q guardrails-ai-detect-pii guardrails-ai-competitor-check guardrails-ai-restrict-to-topic guardrails-ai-toxic-language

In [5]:
import os
os.environ["GUARDRAILS_NO_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

In [6]:
from guardrails.hub import DetectPII
from guardrails.hub import CompetitorCheck
from guardrails.hub import ToxicLanguage
from guardrails.hub import RestrictToTopic
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck

/tmp/ipykernel_660/4133913138.py:1: DeprecationWarning: Importing validators from `guardrails.hub` is deprecated and will be removed in a future major release. Import directly from the `guardrails_ai` namespace instead, e.g. `from guardrails_ai.<name> import DetectPII`.
  from guardrails.hub import DetectPII


In [7]:
pii_guard = Guard().use(
    DetectPII(
        pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        on_fail = OnFailAction.FIX,
        use_local=True,
    )
)

raw="We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com."
out=pii_guard.validate(raw)


COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

# Initialize the guard safely
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)

/usr/local/lib/python3.12/dist-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/spacy/util.py:1800: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


In [8]:
print(out.validated_output)
out = comp_guard.validate(GOOD_TEXT)
print("OUT:", repr(out.validated_output))

We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.
OUT: "Thanks for reaching out! We're happy to help with your refund."


In [9]:
from guardrails.hub import ToxicLanguage

In [10]:
tox_guard = Guard().use(ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail=OnFailAction.FIX, use_local=True,))

In [11]:
rude = "You are completely useless and your stupid app stole my money"
out = tox_guard.validate(rude)
print("validation_passed:", out.validation_passed)
print("OUT:", repr(out.validated_output))

validation_passed: True
OUT: ''


In [12]:
out

ValidationOutcome[TypeVar](call_id='138008547515184', raw_llm_output='You are completely useless and your stupid app stole my money', validation_summaries=[ValidationSummary(validator_name='ToxicLanguage', validator_status='fail', property_path='$', failure_reason='The following sentences in your response were found to be toxic:\n\n- You are completely useless and your stupid app stole my money', error_spans=[ErrorSpan(start=0, end=61, reason='Toxic language detected: toxicity, insult')])], validated_output='', reask=None, validation_passed=True, error=None)

In [13]:
import json, litellm
from guardrails.hub import RestrictToTopic
from guardrails.errors import ValidationError

In [14]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

BOT_MODEL="groq/openai/gpt-oss-20b"

In [15]:
def groq_topic_classifier(text, topics):
    resp = litellm.completion(
        model=BOT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role":"user","content":(
            'Return ONLY a JSON object {"topic_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    try:
        return json.loads(resp.choices[0].message.content).get("topics_present", [])
    except Exception:
        return []

In [16]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,
        disable_llm=False,
        llm_callable=groq_topic_classifier,
        on_fail=OnFailAction.EXCEPTION,
    )
)

Device set to use cpu


In [17]:
off_topic = "Forget Payments - Who do you think will win the next presidential election?"

In [18]:
try:
    out1 = topic_guard.validate(off_topic)
    print("Passed (unexpected):", out1.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print(str(e)[:300])

/usr/local/lib/python3.12/dist-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Raised ValidationError - off-topic refused.
Validation failed for field with errors: No valid topic was found.


# OnFAILACTIONS # 

In [19]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from guardrails.errors import ValidationError

COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and Paypal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

In [20]:
import spacy
import torch

spacy.require_cpu()
torch.set_default_device("cpu")

In [21]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    )

print("Input we'll validate five ways:\n", BAD_TEXT)

Input we'll validate five ways:
 Thanks for reaching out! Honestly, Stripe has lower fees than us, and Paypal is easier to set up. But we're happy to help with your refund.


In [22]:
# Fix OnFailAction

guard=competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)
print("Output Raw response", out)
print("validation_passed:",out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='138009474866768',
    raw_llm_output="Thanks for reaching out! We're happy to help with your refund.",
    validation_summaries=[],
    validated_output="Thanks for reaching out! We're happy to help with your refund.",
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
OUTPUT:
 Thanks for reaching out! We're happy to help with your refund.


In [23]:
# Fix OnFailAction
guard=competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:",out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='138006399501968',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and Paypal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Paypal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: Paypal'
                )
            ]
        )
    ],
    validated_output="Thanks for reaching out!Honestly, [COMPETITOR] has lower fees 

In [24]:
#FILTER ONFAILACTION

guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)
print("OUTPUT RAW RESPONSE", out)
print("VALIDATION_PASSED", out.validation_passed)
print("OUTPUT\n", out.validated_output)

OUTPUT RAW RESPONSE ValidationOutcome[TypeVar](
    call_id='138006398712432',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and Paypal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Paypal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: Paypal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=No

In [25]:
## REFRAIN OnFAILACTION
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)
print("OUTPUT RAW RESPONSE", out)
print("VALIDATION_PASSED", out.validation_passed)
print("OUTPUT\n", out.validated_output)

OUTPUT RAW RESPONSE ValidationOutcome[TypeVar](
    call_id='138006072638784',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and Paypal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Paypal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: Paypal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=No

In [26]:
from typing import Optional
from pydantic import BaseModel, Field

STRUCTURED_JSON = (
    '{"answer":"Refunds settle within 24 hours."'
    ' "comparison":"Honestly Stripe is cheaper than us.",'
    ' "category": "refund"}'
)

In [27]:
def structured_guard(action):
    c = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    class SupportReply(BaseModel):
        answer: str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators":[c]})
        category: str
    return Guard.for_pydantic(SupportReply)

In [28]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"==={action}===")
    print(" After parsing (sentinel placed in the bad field):")
    print(" ", it.parsed_output)
    print(" guarded_output (what the guard actually returns):")
    print(" ",it.guarded_output)
    

===OnFailAction.FILTER===
 After parsing (sentinel placed in the bad field):
  None
 guarded_output (what the guard actually returns):
  None
===OnFailAction.REFRAIN===
 After parsing (sentinel placed in the bad field):
  None
 guarded_output (what the guard actually returns):
  None


In [29]:
#EXCEPTION ONFAILACTION
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_TEXT)
    print("RAW OUTPUT", out)
    print("PASSED (unexpected):", out.validated_output)
except ValidationError as e:
    print("Error RAW OUTPUT", out)
    print("RAISED VALIDATIONERROR")
    print(str(e)[:300])

Error RAW OUTPUT ValidationOutcome[TypeVar](
    call_id='138006072638784',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and Paypal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Paypal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: Paypal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None


In [30]:
#REASK OnFailAction

from email import message
from pyexpat import model

SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model=BOT_MODEL,
    messages=[
        {"role":"system", "content":SYSTEM_PROMPT},
        {"role":"user","content":(
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "Compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2,
)

In [31]:
print(res)

ValidationOutcome[TypeVar](
    call_id='138005548592544',
    raw_llm_output='**Why choose NimbusPay?**\n\n| Feature | NimbusPay | Other Payment Platforms |\n|---------|-----------|------------------------|\n| **Fees** | Lower per‑transaction rates, no monthly fee | Higher per‑transaction fees, optional monthly plans |\n| **Payouts** | Instant or next‑day payouts to bank or wallet | Standard 2‑day payout window (can be faster with instant payouts for a fee) |\n| **Integration** | One‑click SDK for web & mobile, instant card tokenization | Robust API, but requires more setup and maintenance |\n| **Wallet & Loyalty** | Built‑in digital wallet, reward points, and loyalty programs | No native wallet, relies on third‑party solutions |\n| **Customer Experience** | Seamless checkout, single‑click payments, local currency support | Requires redirect or embedded checkout, can feel fragmented |\n| **Security** | PCI‑DSS compliant, 3D Secure 2, tokenization | PCI‑DSS compliant, strong security, 

In [32]:
print("FINAL OUTPUT:\n", res.validated_output)

FINAL OUTPUT:
 **Why choose NimbusPay?**

| Feature | NimbusPay | Other Payment Platforms |
|---------|-----------|------------------------|
| **Fees** | Lower per‑transaction rates, no monthly fee | Higher per‑transaction fees, optional monthly plans |
| **Payouts** | Instant or next‑day payouts to bank or wallet | Standard 2‑day payout window (can be faster with instant payouts for a fee) |
| **Integration** | One‑click SDK for web & mobile, instant card tokenization | Robust API, but requires more setup and maintenance |
| **Wallet & Loyalty** | Built‑in digital wallet, reward points, and loyalty programs | No native wallet, relies on third‑party solutions |
| **Customer Experience** | Seamless checkout, single‑click payments, local currency support | Requires redirect or embedded checkout, can feel fragmented |
| **Security** | PCI‑DSS compliant, 3D Secure 2, tokenization | PCI‑DSS compliant, strong security, but more complex to configure |
| **Support** | 24/7 live chat & dedicate

In [33]:
from typing import Any
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occured:", len(iters) > 1)

for i, it in enumerate[Any](iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips: 2 | Reask occured: True

--- Attempt 1 (status: fail) ---
'**Why choose NimbusPay over Stripe or PayPal?**\n\n| Feature | NimbusPay | Stripe | PayPal |\n|---------|-----------|--------|--------|\n| **Fees** | Lower per‑transaction rates, no monthly fee | Higher per‑transaction fees, optional monthly plans | Higher fees, especially for cross‑border and currency conversion |\n| **Payouts** | Instant or next‑day payouts to your bank or wallet | Standard 2‑day payout window (can be faster with instant payouts for a fee) | 2‑day standard payout, instant payouts for a fee |\n| **Integration** | One‑click SDK for web & mobile, instant card tokenization | Robust API, but requires more setup and maintenance | Simple button integration, but limited customizability |\n| **Wallet & Loyalty** | Built‑in digital wallet, reward points, and loyalty programs | No native wallet, relies on third‑party solutions | PayPal balance exists, but no integrated loyalty or rewards |\n| **Custome

In [34]:
#I/O Structure Validation Using GuardrailsAI
#The Unstructured input

CUSTOMER_EMAIL = (
    "Hi, This is Jane Doe. I was double-charged $49.99 on my Nimbus wallet last Tuesday "
    "When the app froze during checkout. I have been a customer for two years ans this is "
    "Really Frustrating - I need this refunded as soon as possible."
    "My Ticket Should probably go to your billing team."
)
print(CUSTOMER_EMAIL)

Hi, This is Jane Doe. I was double-charged $49.99 on my Nimbus wallet last Tuesday When the app froze during checkout. I have been a customer for two years ans this is Really Frustrating - I need this refunded as soon as possible.My Ticket Should probably go to your billing team.


In [35]:
#Define BluePrint

from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float  = Field(description="Disputed amount in dollars")
    reason: str = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which eam should handle this")
    urgency: Literal["how", "medium", "high"] = Field(
        description="How urgent the request is"
    )

In [36]:
## Extract Structured Data
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)
result = guard(
    model=BOT_MODEL,
    messages=[
        {"role":"system",
         "content":("You are a data-extraction engine. Reply with ONLY a JSON object that matches"
                    "the requested schema. For 'amount', use a plain number with no currency"
                    "Symbol (e.g. 49.99.")},
        {"role":"user",
         "content":"Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed", result.validation_passed)
print("data:", data)

raw output: ValidationOutcome[TypeVar](
    call_id='138005514840864',
    raw_llm_output='{"customer_name":"John Doe","amount":49.99,"reason":"double-charged due to app freeze during checkout","category":"billing","urgency":"high"}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'John Doe',
        'amount': 49.99,
        'reason': 'double-charged due to app freeze during checkout',
        'category': 'billing',
        'urgency': 'high'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed True
data: {'customer_name': 'John Doe', 'amount': 49.99, 'reason': 'double-charged due to app freeze during checkout', 'category': 'billing', 'urgency': 'high'}


In [37]:
if data is None:
    print("No structured output yet - check the DEBUG output in the previous cell to see what the ")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency : {data['urgency']}")

Customer : John Doe
Amount : $49.99
Route to : billing team
Urgency : high


In [39]:
# LangChain X Guardrails AI
!pip install langchain langchain-groq